# 10 · Forecasting with Covariates (XReg)

Real demand depends on **external drivers**: price, promotions, holidays,
day-of-week. TimesFM 2.5 combines a linear covariate model (XReg) with the
foundation model to use them.

> Requires `pip install "timesfm[xreg]"` and `return_backcast=True` in the config.

Covariates come in four kinds:
- **dynamic numerical** — change over time, known into the future (e.g. price)
- **dynamic categorical** — e.g. holiday flag, day-of-week
- **static numerical** — one value per series (e.g. store size)
- **static categorical** — e.g. region

> ⚠️ **Dynamic covariates must span the full length = context + horizon.**

In [ ]:
%pip install -q "timesfm[xreg]" 

In [ ]:
import numpy as np, torch, timesfm
torch.set_float32_matmul_precision("high")

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model.compile(timesfm.ForecastConfig(
    max_context=512, max_horizon=64,
    normalize_inputs=True, use_continuous_quantile_head=True,
    fix_quantile_crossing=True,
    return_backcast=True,      # <-- REQUIRED for covariates
))

In [ ]:
# Build 3 stores, 36 weeks history, forecast 12 weeks.
rng = np.random.default_rng(5)
CTX, H = 36, 12
n_stores = 3

inputs = []                         # sales history (length CTX)
price = []                          # dynamic numeric, length CTX + H
promo = []                          # dynamic categorical, length CTX + H
region = ["north", "south", "north"]  # static categorical, one per store

for s in range(n_stores):
    weeks = np.arange(CTX + H)
    pr = 10 + rng.normal(0, 0.5, CTX + H)           # price path (known ahead)
    pm = (rng.random(CTX + H) < 0.2).astype(int)    # promo weeks
    base = 200 - 8*(pr-10) + 30*pm + 20*np.sin(2*np.pi*weeks/52)
    sales = np.clip(base + rng.normal(0, 6, CTX + H), 0, None)
    inputs.append(sales[:CTX].astype(np.float32))   # history only
    price.append(pr.tolist())
    promo.append(pm.tolist())

print("history length:", len(inputs[0]), "| covariate length:", len(price[0]), "(= CTX + H)")

In [ ]:
point_fc, xreg_fc = model.forecast_with_covariates(
    inputs=inputs,
    dynamic_numerical_covariates={"price": price},
    dynamic_categorical_covariates={"promo": promo},
    static_categorical_covariates={"region": region},
    xreg_mode="xreg + timesfm",   # fit covariates first, TimesFM on residuals
)
for s in range(n_stores):
    print(f"store {s} ({region[s]:>5}): next 4 weeks -> {np.round(point_fc[s][:4], 1)}")

### Two modes
- `"xreg + timesfm"` — fit the covariate model first, then TimesFM forecasts the
  residuals. Good when covariates explain a lot.
- `"timesfm + xreg"` — TimesFM forecasts first, XReg corrects the residuals.

Compare both on your data with the backtesting metrics from notebook 08.